# 🎯 [QLT04_M03] 가동률 저하 상위 5대 요인 도출 1차 통계 분석 Baseline 알고리즘 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m03_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M03 |
| **미션 주제** | 가동률 저하 상위 5대 요인 도출 1차 통계 분석 Baseline 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 가동률 저하 상위 5대 요인 도출 1차 통계 분석 Baseline 알고리즘 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

가동률 저하 상위 요인 도출 1차 통계 분석 Baseline 알고리즘을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **가동률 저하의 상위 5대 요인을 상관·효과크기·손실일로 도출하는 1차 통계 Baseline을 만들고, 정비·검사 병목 판별 룰 15종 이상을 YAML 설정으로 구현해 저가동 후보를 가려낸다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 장비 300대에서 가동률 70% 미만 장비와 나머지의 요인별 평균 차이(효과크기)와 가동률과의 상관을 구해 상위 5개 요인을 정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/engine/qlt04_rule_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 규정 및 지침 조항과의 정확한 일치성

#### 미션 PASS 조건

- 정량 기준: 업무 검증 룰 15종 이상 코딩, 룰 처리 지연시간 50ms 이하


- 진행 기준: 룰 엔진 단위 테스트 100% 통과


- 검증 방법: 경계 조건(Boundary condition)에 대한 룰 판정 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 장비 300대의 가동률 저하 요인을 상관·효과크기·손실일 기준으로 비교하는 1차 통계 Baseline으로 상위 5대 요인을 도출하고, YAML 형식 룰 16종으로 저가동 후보를 판별하며 경계 조건 단위 시험과 룰 처리 지연시간(99백분위)을 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# ---- 정비·검사 병목 판별 룰 설정(YAML 형식 문자열: 기준이 바뀌면 코드가 아니라 이 설정만 고친다) ----
# clause는 룰과 대응시킨 '가상 품질지침 조항' 예시이며 실제 규정 원문이 아니다.
RULES_YAML = """
- {id: Q01, name: 부품 재고 가용률 50% 미만, field: parts_stock_pct, op: "<", value: 50, weight: 2, clause: 가상품질지침 제3조}
- {id: Q02, name: 정비병 충원율 60% 미만, field: mechanic_fill_pct, op: "<", value: 60, weight: 2, clause: 가상품질지침 제4조}
- {id: Q03, name: 부품 대기 8일 초과, field: parts_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제5조}
- {id: Q04, name: 정비병 부족 대기 8일 초과, field: labor_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제6조}
- {id: Q05, name: 외주 정비 15일 초과, field: outsource_days, op: ">", value: 15, weight: 2, clause: 가상품질지침 제7조}
- {id: Q06, name: 외주 검사 대기 6일 초과, field: inspect_wait_days, op: ">", value: 6, weight: 3, clause: 가상품질지침 제8조}
- {id: Q07, name: 재작업 1건 이상, field: rework_count, op: ">=", value: 1, weight: 3, clause: 가상품질지침 제9조}
- {id: Q08, name: 재작업률 25% 초과, field: rework_rate, op: ">", value: 0.25, weight: 2, clause: 가상품질지침 제9조}
- {id: Q09, name: 규격·제원 불일치 3건 이상, field: spec_mismatch, op: ">=", value: 3, weight: 3, clause: 가상품질지침 제10조}
- {id: Q10, name: 하자·불만 3건 이상, field: defect_reports, op: ">=", value: 3, weight: 2, clause: 가상품질지침 제11조}
- {id: Q11, name: 장비 연령 15년 초과, field: age_years, op: ">", value: 15, weight: 1, clause: 가상품질지침 제12조}
- {id: Q12, name: 정비 요청 6건 이상, field: maint_orders, op: ">=", value: 6, weight: 1, clause: 가상품질지침 제13조}
- {id: Q13, name: 대기 일수 합계 30일 초과, field: total_wait, op: ">", value: 30, weight: 3, clause: 가상품질지침 제14조}
- {id: Q14, name: 검사 대기가 외주 정비 일수의 80% 초과, field: insp_ratio, op: ">", value: 0.8, weight: 2, clause: 가상품질지침 제8조}
- {id: Q15, name: 규격 불일치 2건 이상과 하자 접수 동반, field: spec_defect, op: "==", value: true, weight: 3, clause: 가상품질지침 제10조}
- {id: Q16, name: 최근 90일 가동 시간 150시간 미만, field: op_hours, op: "<", value: 150, weight: 1, clause: 가상품질지침 제15조}
"""
GRADE_CUT = [(10, "고위험"), (6, "주의"), (3, "관심")]
OPS = {"<": lambda a, b: a < b, ">": lambda a, b: a > b, "<=": lambda a, b: a <= b, ">=": lambda a, b: a >= b, "==": lambda a, b: a == b}


def parse_rules(text):
    """YAML 한 줄 맵({key: value, ...}) 형식 룰 문자열을 읽는다(외부 yaml 라이브러리 없이)."""
    out = []
    for line in text.strip().splitlines():
        body = line.strip()[3:-1]
        r = {}
        for part in body.split(", "):
            k, v = part.split(": ", 1)
            v = v.strip().strip('"')
            r[k] = True if v == "true" else False if v == "false" else (float(v) if v.replace(".", "", 1).isdigit() else v)
        out.append(r)
    return out


def derive_rec(rec):
    """룰이 쓰는 파생 필드(대기 합계, 재작업률, 검사 대기 비율(외주 5일 이상일 때), 규격 불일치 2건 이상+하자 동반)를 추가한다."""
    r = dict(rec)
    r["total_wait"] = r["parts_wait_days"] + r["labor_wait_days"] + r["outsource_days"] + r["inspect_wait_days"]
    r["rework_rate"] = r["rework_count"] / r["maint_orders"] if r["maint_orders"] else 0.0
    r["insp_ratio"] = r["inspect_wait_days"] / r["outsource_days"] if r["outsource_days"] >= 5 else 0.0
    r["spec_defect"] = bool(r["spec_mismatch"] >= 2 and r["defect_reports"] >= 1)
    return r


def apply_rules(rec, rules, derived=False):
    """룰 적용: (점수, 적중 룰 id 목록). derived=True면 이미 파생 필드가 있는 레코드(경계 시험용)로 본다."""
    r = rec if derived else derive_rec(rec)
    hits = [x["id"] for x in rules if OPS[x["op"]](r[x["field"]], x["value"])]
    w = {x["id"]: x["weight"] for x in rules}
    return sum(w[h] for h in hits), hits


def grade_of(score):
    for cut, g in GRADE_CUT:
        if score >= cut:
            return g
    return "정상"

fleet = make_fleet(300, seed=7)
fleet["low"] = fleet["avail_pct"] < 70                               # 저가동: 가동률 70% 미만(가상 기준)
FAC = ["maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports",
       "age_years", "op_hours", "parts_stock_pct", "mechanic_fill_pct"]
KO = {**FEAT_KO, "parts_wait_days": "부품 대기 일수", "labor_wait_days": "정비병 부족 대기 일수", "op_hours": "가동 시간"}

# 1) 1차 통계 Baseline: 가동률과의 상관계수, 저가동군-정상군 효과크기(Cohen d), 저가동군 평균 값
rows = []
for f in FAC:
    a, b = fleet.loc[fleet["low"], f], fleet.loc[~fleet["low"], f]
    sp = np.sqrt((a.var() + b.var()) / 2) or 1.0
    rows.append({"요인": KO[f], "상관(r)": fleet[f].corr(fleet["avail_pct"]), "효과크기(d)": (a.mean() - b.mean()) / sp, "저가동군 평균": a.mean(), "정상군 평균": b.mean(), "key": f})
fac = pd.DataFrame(rows)
fac["순위점수"] = fac["상관(r)"].abs() + fac["효과크기(d)"].abs() / 3               # 두 지표를 함께 본 순위 점수(휴리스틱)
top5 = fac.sort_values("순위점수", ascending=False).head(5)
print(f"[Baseline] 장비 {len(fleet)}대, 저가동(70% 미만) {int(fleet['low'].sum())}대, 평균 가동률 {fleet['avail_pct'].mean():.1f}%")
print("[가동률 저하 상위 5대 요인]")
print(top5.drop(columns=["key", "순위점수"]).round(2).to_string(index=False))
lost = fleet.loc[fleet["low"], ["parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days"]].mean()
print("저가동 장비의 평균 대기 손실일:", ", ".join(f"{KO[k]} {v:.1f}일" for k, v in lost.items()))
print("주요 저가동 요인 분포(정답):", fleet["main_cause"].value_counts().to_dict())

# 2) 룰 엔진: 장비 레코드에 16종 룰을 적용
rules = parse_rules(RULES_YAML)
recs = fleet.drop(columns=["main_cause", "low"]).to_dict("records")
lat, scores, hits = [], [], []
for r in recs:
    t = time.perf_counter(); s, h = apply_rules(r, rules); lat.append((time.perf_counter() - t) * 1000)
    scores.append(s); hits.append(h)
fleet["score"], fleet["grade"] = scores, [grade_of(s) for s in scores]
print(f"\n[룰 엔진] 룰 {len(rules)}종 적용, 처리 지연 평균 {np.mean(lat):.3f}ms / 99백분위 {np.percentile(lat, 99):.3f}ms (목표 50ms 이하: {'달성' if np.percentile(lat, 99) <= 50 else '미달'})")
tab = fleet.groupby("grade").agg(장비수=("equip_id", "size"), 실제저가동=("low", "sum"), 저가동비율=("low", "mean")).reindex(["고위험", "주의", "관심", "정상"])
print(tab.round(2).to_string())
flag = fleet["score"] >= 6                                                # 주의 이상을 저가동 후보로 본다
tp = int((flag & fleet["low"]).sum()); prec, rec = tp / max(flag.sum(), 1), tp / fleet["low"].sum()
print(f"저가동 후보(주의 이상) {int(flag.sum())}대 중 실제 저가동 {tp}대: 정밀도 {prec:.0%}, 재현율 {rec:.0%}")
cnt = pd.Series([h for hs in hits for h in hs]).value_counts()
print("룰별 적중 상위:", ", ".join(f"{k} {v}건" for k, v in cnt.head(5).items()))

# 3) 경계 조건 단위 시험: 기준값 위·아래에서 룰 판정이 맞는지(룰마다 2~3건)
BASE = derive_rec({"parts_stock_pct": 90, "mechanic_fill_pct": 90, "parts_wait_days": 0, "labor_wait_days": 0, "outsource_days": 0, "inspect_wait_days": 0,
                   "rework_count": 0, "maint_orders": 0, "spec_mismatch": 0, "defect_reports": 0, "age_years": 5, "op_hours": 300})
tests = ok = 0
fails = []
for x in rules:
    v, d = x["value"], (0.01 if isinstance(x["value"], float) and not float(x["value"]).is_integer() else 1)
    cases = {"<": [(v, False), (v - d, True), (v + d, False)], ">": [(v, False), (v + d, True), (v - d, False)], ">=": [(v - d, False), (v, True), (v + d, True)],
             "<=": [(v + d, False), (v, True), (v - d, True)], "==": [(True, True), (False, False)]}[x["op"]]
    for val, expect in cases:
        r = dict(BASE); r[x["field"]] = val
        got = x["id"] in apply_rules(r, rules, derived=True)[1]
        tests += 1; ok += got == expect
        if got != expect:
            fails.append((x["id"], val))
print(f"\n[경계 조건 단위 시험] {ok}/{tests} 통과 ({ok / tests:.0%}), 실패 {fails or '없음'} (목표 100%)")

# 4) 판정 근거 예시(상위 3대)와 사람이 확인할 것
print("\n[고위험 상위 3대 판정 근거]")
nm = {x["id"]: x for x in rules}
for _, r in fleet.sort_values("score", ascending=False).head(3).iterrows():
    h = apply_rules(recs[int(r.name)], rules)[1]
    print(f"  {r['equip_id']} {r['unit']} 점수 {r['score']:.0f} 가동률 {r['avail_pct']}%: " + "; ".join(f"{i} {nm[i]['name']}({nm[i]['clause']})" for i in h[:3]))
print(f"룰 {len(rules)}종(목표 15종 이상: {'달성' if len(rules) >= 15 else '미달'}). 기준값은 YAML만 바꿔 갱신하며, 정비 우선순위와 품질 판정은 정비관·품질검사관이 확인한다.")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
t5 = top5.iloc[::-1]
ax[0].barh([k for k in t5["key"]], t5["상관(r)"], color="tab:blue"); ax[0].set_title("Top-5 factors: correlation with availability")
ax[1].bar(tab.index.map({"고위험": "high", "주의": "caution", "관심": "watch", "정상": "normal"}), tab["저가동비율"], color="tab:orange"); ax[1].set_title("Low-availability rate by rule grade")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[Baseline] 장비 300대, 저가동(70% 미만) 63대, 평균 가동률 80.8%
[가동률 저하 상위 5대 요인]
          요인  상관(r)  효과크기(d)  저가동군 평균  정상군 평균
    정비 요청 건수  -0.75     1.56     3.94    1.95
    외주 정비 일수  -0.73     1.26     8.46    1.78
 외주 검사 대기 일수  -0.66     1.13     5.56    1.02
    부품 대기 일수  -0.63     1.18     8.70    2.48
정비병 부족 대기 일수  -0.57     1.01     8.27    2.59
저가동 장비의 평균 대기 손실일: 부품 대기 일수 8.7일, 정비병 부족 대기 일수 8.3일, 외주 정비 일수 8.5일, 외주 검사 대기 일수 5.6일
주요 저가동 요인 분포(정답): {'없음': 77, '부품 대기': 72, '정비병 부족': 70, '외주 정비': 59, '검사·재작업': 22}

[룰 엔진] 룰 16종 적용, 처리 지연 평균 0.009ms / 99백분위 0.056ms (목표 50ms 이하: 달성)
       장비수  실제저가동  저가동비율
grade                   
고위험     16     13   0.81
주의      24     18   0.75
관심      45     19   0.42
정상     215     13   0.06
저가동 후보(주의 이상) 40대 중 실제 저가동 31대: 정밀도 78%, 재현율 49%
룰별 적중 상위: Q01 53건, Q02 39건, Q03 38건, Q04 28건, Q06 26건

[경계 조건 단위 시험] 47/47 통과 (100%), 실패 없음 (목표 100%)

[고위험 상위 3대 판정 근거]
  E-0047 가상A부대 점수 14 가동률 35.7%: Q01 부품 재고 가용률 50% 미만(가상품질지침 제3조); Q03 부품 대기 8일 초과(가상품질지침 제5조); Q05 외주 정비 15일 초과(가상품질지침 제7조)
  E-0231 가상D부대 점수 14 가동률 51.7%: Q02 정비병 충원율 60% 미만(가상품질지침 제4조); Q03 부품 대기 8일 초과(가상품질지침 제5조); Q04 정비병 부족 대기 8일 초과(가상품질지침 제6조)
  E-0041 가상C부대 점수 13 가동률 38.8%: Q05 외주 정비 15일 초과(가상품질지침 제7조); Q06 외주 검사 대기 6일 초과(가상품질지침 제8조); Q07 재작업 1건 이상(가상품질지침 제9조)
룰 16종(목표 15종 이상: 달성). 기준값은 YAML만 바꿔 갱신하며, 정비 우선순위와 품질 판정은 정비관·품질검사관이 확인한다.
```
</details>
